In [1]:
import os
import numpy as np

from scipy.optimize import linear_sum_assignment

In [2]:
def print_eigenvalues(eigenvalues, name):
    lag = 1  # Assuming lag time of 1 frame
    its = -lag / np.log(np.abs(eigenvalues[1:]))
    print (f'{name}:')
    with np.printoptions(suppress=True): # Suppress scientific notation for better readability
        print(f'\tTop 10 eigenvalues (|λ|): {np.abs(eigenvalues[:10]).round(4)}')
        print(f'\tTop 9 implied timescales (frames): {its[:9].real.round(2)}')

In [4]:
data_path = "/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/data_generation/ala2/solvated/300K"
data_eigenvalues_cg1 = np.load(os.path.join(data_path, "transfer_operator_results_cg1_steps50000_lag1", "eigenvalues.npy"))
data_eigenvalues_cg4 = np.load(os.path.join(data_path, "transfer_operator_results_cg4_steps50000_lag1", "eigenvalues.npy"))

model_path = '/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/train_runs/propagator/tft/cg/gaussian_regression/run_2/predict_runs'
model_mean1 = os.path.join(model_path, "run_variance_by_dimension_mean_1", "Ala2Writer_results")
model_mean_scale2 = os.path.join(model_path, "run_variance_by_dimension_mean_1", "Ala2Writer_results")
model_sample = os.path.join(model_path, "run_variance_by_dimension_sample_1", "Ala2Writer_results")
# model_noise1 = os.path.join(model_path, "run_2", "Ala2Writer_results")
# model_noise2 = os.path.join(model_path, "run_4", "Ala2Writer_results")
# model_nonoise = os.path.join(model_path, "run_3", "Ala2Writer_results")
# model_nonoise_long = os.path.join(model_path, "run_5", "Ala2Writer_results")

model_eigenvalues_mean = np.load(os.path.join(model_mean1, "eigenvalues.npy"))
model_eigenvalues_mean_scale2 = np.load(os.path.join(model_mean_scale2, "eigenvalues.npy"))
model_eigenvalues_sample = np.load(os.path.join(model_sample, "eigenvalues.npy"))

In [5]:
print_eigenvalues(data_eigenvalues_cg1, "MD (No course graining, 50k steps)")
print('-'*100)
print_eigenvalues(data_eigenvalues_cg4, "MD (Course grained, 50k steps)")
print('-'*100)
print_eigenvalues(model_eigenvalues_mean, "Model (Mean noise no scaling, 500k steps)")
print('-'*100)
print_eigenvalues(model_eigenvalues_mean_scale2, "Model (Mean noise with scaling, 500k steps)")
print('-'*100)
print_eigenvalues(model_eigenvalues_sample, "Model (Sample noise, 500k steps)")

MD (No course graining, 50k steps):
	Top 10 eigenvalues (|λ|): [1.     0.9982 0.9607 0.9163 0.8612 0.8326 0.7991 0.7438 0.6951 0.6842]
	Top 9 implied timescales (frames): [557.05  24.94  11.43   6.69   5.46   4.46   3.38   2.75   2.64]
----------------------------------------------------------------------------------------------------
MD (Course grained, 50k steps):
	Top 10 eigenvalues (|λ|): [1.     0.996  0.9033 0.828  0.7402 0.6905 0.6329 0.5814 0.5412 0.5281]
	Top 9 implied timescales (frames): [251.8    9.83   5.3    3.32   2.7    2.19   1.84   1.63   1.57]
----------------------------------------------------------------------------------------------------
Model (Mean noise no scaling, 500k steps):
	Top 10 eigenvalues (|λ|): [0.9986 0.992  0.9815 0.8376 0.8113 0.2933 0.2441 0.1335 0.0821 0.0349]
	Top 9 implied timescales (frames): [123.76  53.52   5.64   4.78   0.82   0.71   0.5    0.4    0.3 ]
---------------------------------------------------------------------------------------

In [ ]:
print(1/(model_eigenvalues_mean.sum().real/data_eigenvalues_cg4.sum().real))
print(1/(model_eigenvalues_mean.sum().real/data_eigenvalues_cg1.sum().real))

2.5517244780536688
3.398692552939812


In [63]:
data_eigenvectors_cg1 = np.load(os.path.join(data_path, "transfer_operator_results_cg1_steps50000_lag1", "eigenvectors.npy"))
data_eigenvectors_cg4 = np.load(os.path.join(data_path, "transfer_operator_results_cg4_steps50000_lag1", "eigenvectors.npy"))
model_eigenvectors_noise1 = np.load(os.path.join(model_noise1, "eigenvectors.npy"))
model_eigenvectors_noise2 = np.load(os.path.join(model_noise2, "eigenvectors.npy"))
model_eigenvectors_nonoise = np.load(os.path.join(model_nonoise, "eigenvectors.npy"))
model_eigenvectors_nonoise_long = np.load(os.path.join(model_nonoise_long, "eigenvectors.npy"))

In [64]:

def calculate_all_angles(eigvecs1, eigvecs2):
    '''
    Calculates the angles between all eigenvectors of two sets. Returns an array of angles in degrees.
    
    Parameters:
    eigvecs1: np.ndarray of shape (n_features, n_vectors_a) - Eigen vectors of the first set
    eigvecs2: np.ndarray of shape (n_features, n_vectors_b) - Eigen vectors of the second set
    
    Returns:
    np.ndarray of shape (n_vectors_a, n_vectors_b) - Angles in degrees between all pairs of eigenvectors
    '''
    angles = np.zeros((eigvecs1.shape[1], eigvecs2.shape[1]))
    for i in range(eigvecs1.shape[1]):
        for j in range(eigvecs2.shape[1]):
            v1 = eigvecs1[:, i].real
            v2 = eigvecs2[:, j].real
            cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))  # Clip to avoid numerical issues
            angles[i, j] = np.degrees(angle)  # Convert to degrees
    
    return np.array(angles)

def get_optimized_angles(eigvecs1, eigvecs2, top_k=None):
    if top_k is None:
        top_k = min(eigvecs1.shape[1], eigvecs2.shape[1])
    angles = calculate_all_angles(eigvecs1[:,:top_k], eigvecs2[:,:top_k])
    row_ind, col_ind = linear_sum_assignment(angles)
    matched_angles = angles[row_ind, col_ind]
    return matched_angles

def calculate_subspace_overlap(eigvecs1, eigvecs2, top_k=None):
    """Calculates the subspace overlap between two sets of eigenvectors using norm of the difference of projection matrices."""
    if top_k is None:
        top_k = min(eigvecs1.shape[1], eigvecs2.shape[1])
    P1 = eigvecs1[:, :top_k] @ eigvecs1[:, :top_k].T
    P2 = eigvecs2[:, :top_k] @ eigvecs2[:, :top_k].T
    overlap = np.linalg.norm(P1 - P2, 'fro')
    return overlap



print('-'*100)
print("Subspace overlap with MD (course grained):")
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_nonoise, data_eigenvectors_cg4)
print("\tModel (No noise):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_nonoise_long, data_eigenvectors_cg4)
print("\tModel (No noise, long):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_noise1, data_eigenvectors_cg4)
print("\tModel (Noise = 1.0):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_noise2, data_eigenvectors_cg4)
print("\tModel (Noise = 2.0):", subspace_overlap)

print('-'*100)
print("Subspace overlap with MD (No course graining):")
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_nonoise, data_eigenvectors_cg1)
print("\tModel (No noise):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_nonoise_long, data_eigenvectors_cg1)
print("\tModel (No noise, long):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_noise1, data_eigenvectors_cg1)
print("\tModel (Noise = 1.0):", subspace_overlap)
subspace_overlap = calculate_subspace_overlap(model_eigenvectors_noise2, data_eigenvectors_cg1)
print("\tModel (Noise = 2.0):", subspace_overlap)

----------------------------------------------------------------------------------------------------
Subspace overlap with MD (course grained):
	Model (No noise): 24.431008973327828
	Model (No noise, long): 25.348532696914525
	Model (Noise = 1.0): 24.31045073262271
	Model (Noise = 2.0): 24.749843439895166
----------------------------------------------------------------------------------------------------
Subspace overlap with MD (No course graining):
	Model (No noise): 24.376414410240432
	Model (No noise, long): 25.415207987954147
	Model (Noise = 1.0): 23.89286640629271
	Model (Noise = 2.0): 24.609898657968298


In [65]:
matched_angles = get_optimized_angles(model_eigenvectors_noise1, data_eigenvectors_cg4)
print("Angles between Model (With noise = 1.0, 50k steps) and MD (Course grained, 50k steps):", matched_angles)

Angles between Model (With noise = 1.0, 50k steps) and MD (Course grained, 50k steps): [89.92827101 89.10612643 88.6315626  89.12477212 88.27077004 89.09504156
 86.88089205 75.90818919 82.54717316 81.81387881 82.05456732 82.7655678
 80.2901552  80.48667597 79.73437686 79.48968486 83.73626222 83.87894992
 75.66446298 73.92923118 79.35059545 79.35059545 74.46357551 76.81193792
 78.18590181 79.74647268 78.63844078 78.63844078 75.55860411 75.55860411
 81.40234526 81.40234526 81.32133943 77.41715147 78.68162055 81.33460209
 79.64588341 79.83039026 80.35095519 76.88861765 76.88861765 81.45631103
 80.27881224 80.51829065 80.51829065 77.77630306 77.15606386 71.53788997
 71.53788997 72.70113392 76.79859296 79.07075248 80.56144346 80.8790004
 77.59565341 77.59565341 76.8165334  79.54921807 75.99975014 75.99975014
 79.37704505 79.40008814 78.11018264 83.95125699 78.15196863 80.2034754
 81.81118501 74.80174817 77.57586007 80.83997815 80.83997815 81.22006934
 81.22006934 78.19538976 78.19538976 80.